# Stage 0 latency calibration — JupyterLab A100

Local equivalent of `modal run modal_app.py::main --command stage0`, for a GPU box
without Modal. Mirrors `Dockerfile.modal`; does not touch the Modal wiring.

Companion to `docs/RUNNING_ON_A100_JUPYTER.md`, which has the same steps as shell
commands plus more troubleshooting.

## Do not use "Run All"

Run the cells one at a time. There are three points where the notebook cannot
continue on its own:

1. **Section 3 — restart the kernel.** The kernel started before `lerobot`/`libero`
   were installed and cannot see them until it restarts.
2. **Section 6 — paste your HuggingFace token.** The cell ships a placeholder.
3. **Section 9 launches a 5-9 hour run and returns immediately.** Section 11 must
   not be run until section 10 reports every episode complete; it raises if you try.

**The sweep is launched detached, not hosted in a cell.** A cell that owns the
process dies with the kernel — browser disconnect, kernel restart, or a recycled
JupyterLab session all lose the run *and* the GPU-hours. Section 9 uses
`start_new_session=True` so the job outlives the kernel; section 10 is a re-runnable
progress check that works after a restart.


## 1. Repo

In [ ]:
import subprocess
from pathlib import Path

REPO = Path.home() / "async-vla-latency-bench"
URL = "https://github.com/MaheshGouru/async-vla-latency-bench.git"

if REPO.exists():
    print("already present:", REPO)
else:
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)


In [ ]:
%cd ~/async-vla-latency-bench
!git checkout mathew_branch
!git log --oneline -1

## 2. Check the box

`d*` depends on measured inference latency, so the GPU matters. The frozen grid
(`n_action_steps=25`, delays to +400 ms) was sized against an **A100-SXM4-40GB**
measuring ~500 ms naive / ~735 ms RTC per request. Different silicon shifts those,
and section 7 checks whether the sizing still holds.

In [ ]:
!nvidia-smi
!python3 --version
!df -h ~ | tail -1

In [ ]:
# Headless MuJoCo needs EGL. If this raises, install the mesa/EGL packages listed in
# docs/RUNNING_ON_A100_JUPYTER.md section 1, or fall back to osmesa in section 4.
import ctypes
ctypes.CDLL("libEGL.so.1")
print("EGL ok")

## 3. Install

`%pip` (not `!pip`) targets the kernel's own interpreter, avoiding the
kernel/shell mismatch trap.

In [ ]:
%pip install -e .

In [ ]:
# Pinned to the same SHA the Modal image builds from. Do not use "main": measured
# inference latency is the independent variable here, so an unpinned LeRobot can
# move d* between runs (this is what commit da690e6 pinned for Modal).
LEROBOT_COMMIT = "2aba372b4e217cc47db28e0f836859b20d1456c9"
%pip install "lerobot[pi,libero] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_COMMIT}"

### RESTART KERNEL now

The kernel started before `lerobot`/`libero` existed, so it cannot see them yet.
**Kernel -> Restart Kernel**, then continue from the next cell. You do not need to
re-run anything above.

In [ ]:
%cd ~/async-vla-latency-bench
import libero
print(libero.__file__)

## 4. Rendering environment

In [ ]:
import os

# Matches Dockerfile.modal lines 6-10. Use "osmesa" for both if EGL is unavailable
# (software rendering: works, but roughly 40% slower).
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
print(os.environ["MUJOCO_GL"])

## 5. Pre-create `~/.libero/config.yaml`

`libero.libero.__init__` calls `input()` on first import if this file is missing.
In a cell that hangs the kernel; under a detached process it hangs forever. Only
top-level `import libero` is safe before this runs — never `import libero.libero`.

In [ ]:
import os, yaml, libero

root = os.path.join(os.path.dirname(libero.__file__), "libero")
cfg = {
    "benchmark_root": root,
    "bddl_files": os.path.join(root, "bddl_files"),
    "init_states": os.path.join(root, "init_files"),
    "datasets": os.path.join(root, "../datasets"),
    "assets": os.path.join(root, "assets"),
}
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
with open(os.path.expanduser("~/.libero/config.yaml"), "w") as fh:
    yaml.dump(cfg, fh)
print(cfg["benchmark_root"])

## 6. HuggingFace token and paths

Replaces the Modal secret `hf-token`. **Clear this cell's value before committing
the notebook.**

In [ ]:
import os
os.environ["HF_TOKEN"] = "<your_hf_token>"

In [ ]:
from pathlib import Path

CONFIG = "async_vla_benchmark/configs/stage0.yaml"
OUTPUT_DIR = Path("outputs/stage0")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_LOG = OUTPUT_DIR / "stage0_run.log"
PID_FILE = OUTPUT_DIR / "stage0.pid"
RESULTS = OUTPUT_DIR / "latency_calibration_episode_results.csv"

from async_vla_benchmark.benchmark.stage0 import (
    ADDED_DELAYS_MS, EXECUTION_METHODS, FIXED_HORIZON,
    REQUEST_THRESHOLD_ACTIONS, SEEDS, STAGE0_TASKS, stage0_manifest,
)
print("horizon        ", FIXED_HORIZON)
print("threshold      ", REQUEST_THRESHOLD_ACTIONS)
print("delays (ms)    ", ADDED_DELAYS_MS)
print("seeds          ", SEEDS)
print("planned episodes", len(stage0_manifest()))

## 7. Preflight

Asserts resolved task names against the live suite, control frequency,
`n_action_steps`, the `chunk_size >= 2 x H` invariant the delay grid depends on,
and that RTC is actually active on the policy. The first run downloads the pi05
checkpoint and ~586 LIBERO asset files, so expect several quiet minutes.

In [ ]:
!python -m async_vla_benchmark.scripts.run_stage0 --config {CONFIG} --output-dir {OUTPUT_DIR} --preflight-only

In [ ]:
# Confirm the grid before spending GPU-hours on it.
!python -m async_vla_benchmark.scripts.run_stage0 --config {CONFIG} --output-dir {OUTPUT_DIR} --dry-run | tail -3

## 7b. Pick a free GPU

The box has 8 A100s in `Exclusive_Process` mode: one process per device, and
most are usually taken by other users. The kernel starts pinned to whichever
`CUDA_VISIBLE_DEVICES` says, which is often an occupied one — that surfaces as
`cudaErrorDevicesUnavailable` when the policy is moved to the GPU.

Exclusive mode works in your favour once you have a device: no other user can
take it mid-run, so a 5-9 hour sweep is safe from eviction.


In [ ]:
import os, subprocess

# nvidia-smi reports all devices regardless of CUDA_VISIBLE_DEVICES, so this sees
# the whole box even though the kernel is currently pinned to one GPU.
out = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,memory.used,memory.total",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True,
).stdout.strip().splitlines()

gpus = []
for line in out:
    idx, used, total = (int(x) for x in line.split(","))
    gpus.append((used, idx, total))
gpus.sort()

print("GPU   used / total MiB")
for used, idx, total in sorted(gpus, key=lambda g: g[1]):
    print(f"  {idx}   {used:>6} / {total}   {'FREE' if used < 500 else ''}")

used, idx, _ = gpus[0]
if used >= 500:
    raise SystemExit(
        f"no free GPU: emptiest is {idx} with {used} MiB in use. Every device is "
        "Exclusive_Process, so wait for one to clear rather than trying to share."
    )

# Must be set in Python. `!export CUDA_VISIBLE_DEVICES=...` runs in a subshell that
# exits immediately and never reaches this process, so it silently does nothing --
# every later cell would keep using whichever GPU the kernel started pinned to.
os.environ["CUDA_VISIBLE_DEVICES"] = str(idx)

# MuJoCo's EGL backend selects its device independently of CUDA_VISIBLE_DEVICES and
# defaults to physical device 0. robosuite builds its render context before the
# policy loads, so if device 0 is held by another user under Exclusive_Process, that
# poisons the process's CUDA state and the later .to("cuda") fails with
# cudaErrorDevicesUnavailable -- even when the GPU you asked for is completely free.
# This takes the PHYSICAL index, while CUDA sees the same card as cuda:0.
os.environ["MUJOCO_EGL_DEVICE_ID"] = str(idx)

print(f"\nCUDA_VISIBLE_DEVICES={idx}  (appears as cuda:0 inside the run)")
print(f"MUJOCO_EGL_DEVICE_ID={idx}  (physical index)")


## 8. Latency check — does the frozen horizon still fit this GPU?

The queue holds `H` actions draining at one per control step, so a chunk covers
`H x period` of robot time. That is the total request latency it can absorb before
underrunning and holding position. Utilisation is

    rho = (drain before the request goes out + total latency) / (H x period)

`rho >= 1` means the queue empties every cycle and the arm freezes `1 - 1/rho` of all
steps — which is what made the first calibration (`H=10`, rho ~ 1.5 at zero added
delay) measure starvation instead of action staleness.

This measures native latency on *this* box and re-derives the table.

In [ ]:
!python -m async_vla_benchmark.scripts.profile_latency --config {CONFIG} --output-dir {OUTPUT_DIR} --measured-requests 50

In [ ]:
import json

native = json.load(open(OUTPUT_DIR / "summaries" / "native_latency.json"))
hz = native["control_frequency_hz"]
period_ms = 1000.0 / hz

naive_ms = native["mean_ms"]
# profile_latency measures a plain forward pass. RTC's guided decode ran ~1.4x that
# on the A100-40GB reference run (~500 ms -> ~700 ms); rescale rather than remeasure.
RTC_OVERHEAD = 1.4
rtc_ms = naive_ms * RTC_OVERHEAD

budget_ms = FIXED_HORIZON * period_ms
drain_ms = (FIXED_HORIZON - REQUEST_THRESHOLD_ACTIONS) * period_ms

print(f"control {hz:g} Hz -> {period_ms:.1f} ms/step")
print(f"measured naive inference {naive_ms:.0f} ms   (reference box: ~500 ms)")
print(f"estimated RTC inference  {rtc_ms:.0f} ms   (reference box: ~735 ms)")
print(f"chunk budget = {FIXED_HORIZON} x {period_ms:.0f} = {budget_ms:.0f} ms\n")

worst = 0.0
for label, infer in (("naive", naive_ms), ("rtc", rtc_ms)):
    for d in ADDED_DELAYS_MS:
        rho = (drain_ms + infer + d) / budget_ms
        worst = max(worst, rho)
        flag = "   <-- STARVES" if rho >= 1 else ""
        print(f"  {label:>5} +{d:<4} rho={rho:.2f}{flag}")

print()
if worst >= 1.0:
    raise SystemExit(
        f"STOP: worst rho = {worst:.2f}. This box is slower than the one the grid was "
        "sized on, so the queue would starve and the sweep would measure underruns "
        "instead of action staleness -- the exact defect that invalidated the H=10 "
        "calibration. Raising FIXED_HORIZON or shortening ADDED_DELAYS_MS is a change "
        "to D002/D007: amend the decisions first, do not just edit the constants."
    )
if worst > 0.9:
    print(f"Tight: worst rho = {worst:.2f}. Runnable, but little margin for latency spikes.")
else:
    print(f"OK: worst rho = {worst:.2f}. Queue stays ahead across the whole grid.")

## 9. Launch the sweep (detached)

`start_new_session=True` puts the run in its own session, so it survives kernel
restarts and browser disconnects. The cell returns immediately.

`--resume` skips episodes already in the results CSV, so re-running this cell after
a crash or a session limit costs at most one episode. Safe to re-run.

In [ ]:
import os, subprocess, sys

if PID_FILE.exists():
    old = int(PID_FILE.read_text().strip())
    try:
        os.kill(old, 0)
        raise SystemExit(f"a run is already alive with pid {old} -- check section 10 first")
    except ProcessLookupError:
        print(f"stale pid {old}, continuing")

# -u / PYTHONUNBUFFERED: with stdout redirected to a file Python block-buffers in
# ~4-8 KB chunks, so the log can look frozen for many minutes while the run is
# perfectly healthy -- which makes it useless as a progress signal. The Modal image
# set PYTHONUNBUFFERED=1 for the same reason.
env = os.environ.copy()          # carries CUDA_VISIBLE_DEVICES / MUJOCO_* / HF_TOKEN
env["PYTHONUNBUFFERED"] = "1"

log = open(RUN_LOG, "ab")
proc = subprocess.Popen(
    [sys.executable, "-u", "-m", "async_vla_benchmark.scripts.run_stage0",
     "--config", CONFIG, "--output-dir", str(OUTPUT_DIR), "--resume"],
    stdout=log, stderr=subprocess.STDOUT,
    start_new_session=True,      # detach: outlives this kernel
    env=env,
)
PID_FILE.write_text(str(proc.pid))
print("launched pid", proc.pid, "->", RUN_LOG)

## 10. Progress

Re-run this cell whenever. It reads the log and results file, not the process, so it
works after a kernel restart.

In [ ]:
import os, time
from pathlib import Path

# Progress comes from the per-episode artifacts, NOT the results CSV: run_stage0
# writes latency_calibration_episode_results.csv once at the very end, so counting
# its rows reports 0 for the entire run. episodes/*.json is what --resume reads.
EPISODES = OUTPUT_DIR / "episodes"

pid = int(PID_FILE.read_text().strip()) if PID_FILE.exists() else None
alive = False
if pid:
    try:
        os.kill(pid, 0); alive = True
    except (ProcessLookupError, PermissionError):
        alive = False

planned = len(stage0_manifest())
done = len(list(EPISODES.glob("*.json"))) if EPISODES.exists() else 0

print(f"pid {pid}  alive={alive}")
print(f"episodes {done}/{planned}  ({100.0 * done / planned:.0f}%)")

if done and RUN_LOG.exists():
    per = (time.time() - RUN_LOG.stat().st_ctime) / done
    print(f"~{per:.0f}s/episode   eta {(planned - done) * per / 3600:.1f} h")

print("\n--- tail ---")
print("".join(open(RUN_LOG).readlines()[-12:]) if RUN_LOG.exists() else "(no log yet)")


In [ ]:
# Stop the run if you need to. --resume picks it back up from where it stopped.
# import os, signal; os.kill(int(PID_FILE.read_text()), signal.SIGTERM)

## 11. Analyse

Only once section 10 shows every episode complete. `select_high_delay` writes
`selected_high_delay.json`, which Stage 1 reads instead of choosing its own delay.

In [ ]:
from pathlib import Path

# Same source as the progress cell: the results CSV does not exist until the sweep
# has fully finished, so completeness is judged from the per-episode artifacts.
planned = len(stage0_manifest())
done = len(list((OUTPUT_DIR / "episodes").glob("*.json")))
if done < planned:
    raise SystemExit(
        f"only {done}/{planned} episodes complete -- wait for section 10 to finish. "
        "d* is frozen from this curve and inherited by every later stage, so it must "
        "not be selected from a partial grid."
    )
print(f"{done}/{planned} episodes; proceeding")


In [ ]:
!python -m async_vla_benchmark.scripts.validate_results --output-dir {OUTPUT_DIR}


In [ ]:
!python -m async_vla_benchmark.scripts.select_high_delay --results {RESULTS}

In [ ]:
import json

sel = json.load(open(OUTPUT_DIR / "selected_high_delay.json"))
print(f"d*                  {sel['high_added_delay_ms']} ms")
print(f"rule applied        {sel.get('rule_applied')}")
print(f"native pooled       {sel.get('native_success_pooled')}")
print(f"selected pooled     {sel.get('selected_success_pooled')}")
print(f"saturated / weak    {sel['calibration_saturated']} / {sel['calibration_weak']}")
print(f"viable cells        {sel.get('viable_cells')}")
for note in sel.get("notes", []):
    print("  note:", note)

In [ ]:
# Record the environment alongside the results: on Modal the LeRobot pin is enforced
# by the image, here only by the install command in section 3.
!python -m pip freeze > {OUTPUT_DIR}/stage0_pip_freeze.txt
!wc -l {OUTPUT_DIR}/stage0_pip_freeze.txt

## Troubleshooting

**`import` fails right after `%pip install`** — restart the kernel before assuming the
install failed; the running kernel caches the package listing from before the install.

**Run dies at the same episode every time** — check the tail of `stage0_run.log`. A
per-episode exception is recorded with `status=invalid` and the sweep continues, so a
hard stop is usually the environment (OOM, GL, disk), not one bad episode.

**Session recycled mid-run** — expected on shared allocations. Re-run section 9; it
resumes. Episode order is task -> method -> delay -> seed, so a partial run always has
whole task x method cells finished, which is what viability is computed over.

In [ ]:
import site, sys
print("executable:", sys.executable)
print([p for p in sys.path if "site-packages" in p])